# Mini framework numpy, MNIST ≥ 97.5%

Bạn dựng một framework nhỏ kiểu PyTorch bằng numpy: `Module`/`Sequential`, `Linear`, `ReLU`, `BatchNorm1d`,
`Dropout`, `SoftmaxCrossEntropy`, `AdamW`, cosine schedule, early stopping; rồi train một MLP trên MNIST tới
**test accuracy ≥ 97.5%** và viết `REPORT.md` như một bài portfolio.

**Cách làm bài**
- API dưới đây là **gợi ý**: bạn được đổi thiết kế (tên lớp, thuộc tính, thêm layer). Test chỉ gọi ba hàm
  `build_model(seed)`, `fit(model, X, y, X_val, y_val, epochs)`, `evaluate(model, X, y)`.
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu, train,
  vẽ; test bỏ qua chúng.
- Quy ước như phần lý thuyết: batch theo dòng, `W` shape `(in, out)`, $$\text{out} = xW + b$$.
- Chấm: `uv run pytest tracks/ai-engineer/modules/01-nn-from-scratch/exercises/advanced/01-numpy-framework-mnist -q`

In [ ]:
# explore: giới hạn luồng BLAS, tìm gốc repo, tải MNIST nếu chưa có (khoảng 11 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy; ma trận nhỏ mà 16 luồng thì chậm hơn ~5 lần
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

MNIST = ensure("mnist")
print(MNIST)

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import gzip
import time
from pathlib import Path

import numpy as np

In [ ]:
def load_mnist(root: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Đọc 4 tệp idx gz → (X_train, y_train, X_test, y_test); X float32 (N, 784) trong [0, 1], y int64.

    Dùng lại hàm của bài Intermediate 01.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc MNIST, tách val = 10 000 ảnh cuối của tập train (test chỉ dùng một lần ở cuối)
X_all, y_all, X_test, y_test = load_mnist(MNIST)
X_train, y_train, X_val, y_val = X_all[:50000], y_all[:50000], X_all[50000:], y_all[50000:]
print(X_train.shape, X_val.shape, X_test.shape)

## 1. Lõi: Param, Module, Sequential

Mỗi layer là một `Module` với `forward(x)`, `backward(grad)` (nhận $$\partial L/\partial \text{out}$$, ghi gradient
vào `Param` của mình, trả $$\partial L/\partial x$$), `parameters()`, `train()`/`eval()`. `Param` gom giá trị,
gradient và cờ `decay`: weight decay thường chỉ áp cho ma trận trọng số, không cho bias và `gamma`/`beta` của
BatchNorm. `state()` trả **mọi** mảng cần chụp lại khi lưu trọng số tốt nhất: BatchNorm có running stats
không phải tham số nhưng vẫn phải lưu.

In [ ]:
class Param:
    """Một tham số học được: `data` (giá trị), `grad` (cùng shape), `decay` (có áp weight decay không)."""

    def __init__(self, data: np.ndarray, decay: bool = True):
        # TODO
        raise NotImplementedError


class Module:
    """Lớp gốc. Mọi layer có forward/backward, danh sách Param, và chế độ train/eval."""

    training = True

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.forward(x)

    def forward(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def backward(self, grad: np.ndarray) -> np.ndarray:
        """Nhận dL/d(output), ghi gradient vào Param của mình, trả dL/d(input)."""
        raise NotImplementedError

    def parameters(self) -> list[Param]:
        # TODO: layer không có tham số trả []
        raise NotImplementedError

    def state(self) -> list[np.ndarray]:
        """Mọi mảng cần chụp lại khi lưu "trọng số tốt nhất" (tham số và buffer như running stats)."""
        # TODO
        raise NotImplementedError

    def train(self, mode: bool = True) -> "Module":
        # TODO
        raise NotImplementedError

    def eval(self) -> "Module":
        return self.train(False)


class Sequential(Module):
    def __init__(self, *layers: Module):
        # TODO: forward theo thứ tự, backward ngược lại; parameters/state/train lan xuống mọi layer
        raise NotImplementedError

## 2. Layer

Lấy lại code của bài Basic 03 (Linear, ReLU) và Intermediate 03 (BatchNorm1d, Dropout), viết theo API trên.
`Sequential.train(mode)` phải lan xuống mọi layer con, vì BatchNorm và Dropout hành xử khác nhau giữa train và eval.

In [ ]:
class Linear(Module):
    """out = x @ W + b, W shape (n_in, n_out) khởi tạo He, float32."""

    def __init__(self, n_in: int, n_out: int, rng: np.random.Generator):
        # TODO
        raise NotImplementedError


class ReLU(Module):
    # TODO
    pass


class BatchNorm1d(Module):
    """Như bài Intermediate 03: batch stats khi train, running stats khi eval."""

    def __init__(self, n: int, momentum: float = 0.1, eps: float = 1e-5):
        # TODO
        raise NotImplementedError


class Dropout(Module):
    """Inverted dropout; eval thì trả nguyên input."""

    def __init__(self, p: float, rng: np.random.Generator):
        # TODO
        raise NotImplementedError

## 3. Loss, optimizer, schedule, early stopping

- `SoftmaxCrossEntropy`: loss trung bình trên batch, `backward()` trả $$(p - \text{onehot}(y)) / N$$.
- `AdamW` như bài Intermediate 02 nhưng làm việc trên `Param`; `SGDMomentum` để so trong ablation.
- `cosine_lr`: cập nhật lr **theo bước**, không theo epoch.
- `EarlyStopping`: chụp `state()` của model mỗi khi val accuracy tốt hơn, `restore` khi kết thúc.

In [ ]:
class SoftmaxCrossEntropy:
    def forward(self, logits: np.ndarray, y: np.ndarray) -> float:
        """Cross-entropy trung bình trên batch, log-softmax ổn định số."""
        # TODO
        raise NotImplementedError

    def backward(self) -> np.ndarray:
        """dL/d(logits) của lần forward trước."""
        # TODO
        raise NotImplementedError

In [ ]:
class AdamW:
    """AdamW trên danh sách Param; weight decay chỉ áp cho Param có `decay=True`."""

    def __init__(self, params: list[Param], lr: float = 1e-3, betas=(0.9, 0.999), eps: float = 1e-8,
                 weight_decay: float = 0.0):
        # TODO
        raise NotImplementedError

    def step(self) -> None:
        # TODO
        raise NotImplementedError


class SGDMomentum:
    """Để so optimizer trong ablation (lấy lại bài Intermediate 02)."""

    def __init__(self, params: list[Param], lr: float = 0.1, momentum: float = 0.9):
        # TODO
        raise NotImplementedError

    def step(self) -> None:
        # TODO
        raise NotImplementedError


def cosine_lr(step: int, total: int, lr_max: float, warmup: int = 0, lr_min: float = 0.0) -> float:
    """Learning rate ở bước `step` (đếm từ 0) trên tổng `total` bước: warmup tuyến tính rồi cosine decay."""
    # TODO
    raise NotImplementedError


class EarlyStopping:
    """Theo dõi val accuracy; chụp `model.state()` khi tốt hơn; báo dừng khi quá `patience` epoch không tốt hơn."""

    def __init__(self, patience: int = 3):
        # TODO
        raise NotImplementedError

    def update(self, score: float, model: Module) -> bool:
        """Ghi nhận điểm val của epoch vừa xong; True nếu nên dừng."""
        # TODO
        raise NotImplementedError

    def restore(self, model: Module) -> None:
        """Chép bản chụp tốt nhất vào chính các mảng của model."""
        # TODO
        raise NotImplementedError

In [ ]:
# explore: gradient check cả mạng nhỏ (Linear, BatchNorm1d ở chế độ train, ReLU, loss), float64
def check_gradients(model: Module, X: np.ndarray, y: np.ndarray, eps: float = 1e-6) -> float:
    """max|giải tích - số| / (max|giải tích| + max|số|) trên gradient của mọi Param ghép lại."""
    loss_fn = SoftmaxCrossEntropy()
    for p in model.parameters():                    # float64 để sai phân hữu hạn đủ chính xác
        p.data = p.data.astype(np.float64)
        p.grad = np.zeros_like(p.data)
    model.train()
    loss_fn.forward(model(X), y)
    model.backward(loss_fn.backward())
    analytic_all, numeric_all = [], []
    for p in model.parameters():
        analytic, numeric = p.grad.copy(), np.zeros_like(p.data)
        for i in np.ndindex(p.data.shape):
            old = p.data[i]
            p.data[i] = old + eps
            lp = loss_fn.forward(model(X), y)
            p.data[i] = old - eps
            lm = loss_fn.forward(model(X), y)
            p.data[i] = old
            numeric[i] = (lp - lm) / (2 * eps)
        analytic_all.append(analytic.ravel())
        numeric_all.append(numeric.ravel())
    a, n = np.concatenate(analytic_all), np.concatenate(numeric_all)
    return float(np.abs(a - n).max() / (np.abs(a).max() + np.abs(n).max()))

rng = np.random.default_rng(0)
tiny = Sequential(Linear(6, 5, rng), BatchNorm1d(5), ReLU(), Linear(5, 3, rng))
Xg, yg = rng.normal(size=(8, 6)), rng.integers(0, 3, 8)
# Đo trên cả vector gradient, không chia từng phần tử: bias của Linear ngay trước BatchNorm có gradient đúng
# bằng 0 (BN trừ mean), chia cho một số sát 0 thì sai lệch tương đối vô nghĩa.
print(f"sai lệch tương đối: {check_gradients(tiny, Xg, yg):.1e} (nên dưới 1e-6)")

## 4. Ba hàm được chấm

- `build_model(seed, **options)`: model 784 → 10 logits, mọi số ngẫu nhiên từ `seed`.
- `fit(model, X, y, X_val, y_val, epochs, **options) -> {"train_loss": [...], "val_acc": [...]}`: mỗi phần tử
  một epoch đã chạy (dừng sớm thì ít hơn `epochs`); kết thúc với trọng số của epoch val tốt nhất, chế độ eval.
- `evaluate(model, X, y) -> float`: accuracy ở chế độ eval, kết quả không phụ thuộc cách chia batch.

Test thật: `fit(build_model(0), 50 000 ảnh train đầu, 10 000 ảnh còn lại làm val, epochs=10)` phải đạt test
accuracy ≥ 0.975 trong dưới 4 phút.

In [ ]:
def build_model(seed: int, **options) -> Module:
    """Model cho MNIST: input 784, output 10 logits. Mọi số ngẫu nhiên (init, dropout) lấy từ `seed`.

    `options`: tuỳ chọn có mặc định của bạn (vd. bn, dropout, hidden) để chạy ablation.
    """
    # TODO
    raise NotImplementedError


def fit(model: Module, X: np.ndarray, y: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, epochs: int,
        **options) -> dict[str, list[float]]:
    """Train tối đa `epochs` epoch; mỗi epoch ghi train loss (trung bình theo mẫu) và val accuracy.

    Có early stopping; khi kết thúc, model mang trọng số của epoch có val accuracy cao nhất, ở chế độ eval.
    Tái lập: cùng `build_model(seed)` và cùng dữ liệu cho cùng history.
    `options`: tuỳ chọn có mặc định (lr, optimizer, schedule, patience, batch_size...) cho ablation.
    """
    # TODO
    raise NotImplementedError


def evaluate(model: Module, X: np.ndarray, y: np.ndarray) -> float:
    """Accuracy trong [0, 1], ở chế độ eval, không đổi trạng thái của model."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: train model chính, vẽ đường cong, lưu hình cho REPORT.md
import matplotlib.pyplot as plt

start = time.perf_counter()
model = build_model(0)
history = fit(model, X_train, y_train, X_val, y_val, epochs=15)
print(f"{len(history['val_acc'])} epoch, {time.perf_counter() - start:.0f} s | best val {max(history['val_acc']):.4f}"
      f" | test {evaluate(model, X_test, y_test):.4f}")

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
epochs = range(1, len(history["train_loss"]) + 1)
a.plot(epochs, history["train_loss"], marker="o"); a.set_xlabel("epoch"); a.set_ylabel("train loss"); a.set_yscale("log")
b.plot(epochs, history["val_acc"], marker="o"); b.set_xlabel("epoch"); b.set_ylabel("val accuracy")
fig.tight_layout()
fig.savefig("curves.png", dpi=120)                  # cạnh notebook, để REPORT.md chèn ![](curves.png)
plt.show()

In [ ]:
# explore: ablation, mỗi cấu hình 2 seed, 10 epoch (khoảng 5 phút). Đổi tên khoá cho khớp `options` của bạn.
CONFIGS = {
    "đầy đủ": ({}, {}),
    "bỏ BatchNorm": ({"bn": False}, {}),
    "bỏ dropout": ({"dropout": 0.0}, {}),
    "lr cố định (không cosine)": ({}, {"schedule": "constant"}),
    "SGD momentum thay AdamW": ({}, {"optimizer": "sgd", "lr": 0.05}),
}
print("| Cấu hình | Test acc seed 0 / 1 | Best val | Epoch | Thời gian |")
print("| --- | --- | --- | --- | --- |")
for name, (model_opts, fit_opts) in CONFIGS.items():
    accs, vals, n_ep, start = [], [], [], time.perf_counter()
    for seed in [0, 1]:
        m = build_model(seed, **model_opts)
        h = fit(m, X_train, y_train, X_val, y_val, epochs=10, **fit_opts)
        accs.append(evaluate(m, X_test, y_test)); vals.append(max(h["val_acc"])); n_ep.append(len(h["val_acc"]))
    print(f"| {name} | {accs[0]:.4f} / {accs[1]:.4f} | {np.mean(vals):.4f} | {n_ep} | {time.perf_counter() - start:.0f} s |")

## 5. Báo cáo

Điền `REPORT.md` (cùng thư mục) theo khung: kiến trúc framework, đường cong (`curves.png`), bảng ablation,
lỗi đã gặp và cách bạn phát hiện. Rồi làm phần **trắc nghiệm** của bài trong app.